# TextCNN 情感分类

先准备训练数据，并用一个批次检查模型的训练步骤。测试集留到最终评估。

In [ ]:
# 加载训练集和验证集
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from sentiment_analysis.paths import CLEANED_DIR
from sentiment_analysis.sequence_data import build_vocabulary, SentimentDataset
from sentiment_analysis.textcnn import TextCNN

train_data = pd.read_csv(CLEANED_DIR / "train.csv")
validation_data = pd.read_csv(CLEANED_DIR / "val.csv")

print(f"Train: {len(train_data):,}")
print(f"Validation: {len(validation_data):,}")

In [ ]:
# 使用训练集建立词表和数据加载器
max_words = 20_000
max_length = 512
batch_size = 32
random_seed = 42

vocabulary = build_vocabulary(train_data["text"], max_words=max_words)

train_dataset = SentimentDataset(train_data, vocabulary, max_length)
validation_dataset = SentimentDataset(validation_data, vocabulary, max_length)

generator = torch.Generator().manual_seed(random_seed)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, generator=generator,
)
validation_loader = DataLoader(
    validation_dataset, batch_size=batch_size, shuffle=False,
)

print(f"Vocabulary size: {len(vocabulary):,}")
print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(validation_loader):,}")

## 单批次训练检查

先用临时模型验证前向计算、损失计算、反向传播和参数更新。正式训练时将重新初始化模型。

In [ ]:
# 初始化临时模型、损失函数和优化器
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(random_seed)

example_model = TextCNN(
    vocabulary_size=len(vocabulary),
    embedding_dim=128,
    num_filters=64,
    kernel_size=3,
    padding_index=vocabulary["<PAD>"],
).to(device)

loss_function = nn.CrossEntropyLoss()
example_optimizer = torch.optim.Adam(example_model.parameters(), lr=0.001)

print("Device:", device)

In [ ]:
# 使用一个批次检查参数更新
batch_token_ids, batch_lengths, batch_labels = next(iter(train_loader))
batch_token_ids = batch_token_ids.to(device)
batch_lengths = batch_lengths.to(device)
batch_labels = batch_labels.to(device)

example_model.train()
weights_before = example_model.classifier.weight.detach().clone()

example_optimizer.zero_grad()
batch_logits = example_model(batch_token_ids, batch_lengths)
batch_loss = loss_function(batch_logits, batch_labels)
batch_loss.backward()
example_optimizer.step()

weights_changed = not torch.equal(weights_before, example_model.classifier.weight.detach())

print("Output shape:", batch_logits.shape)
print(f"Batch loss: {batch_loss.item():.4f}")
print("Classifier weights updated:", weights_changed)

# 重置训练集打乱顺序，正式训练时重新初始化模型和优化器
generator.manual_seed(random_seed)

## 训练与验证流程

训练时更新模型参数，验证时关闭 Dropout 和梯度计算。损失与准确率均按样本数量计算。

In [ ]:
# 定义一个训练轮次
def train_one_epoch(model, data_loader, optimizer, loss_function, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for token_ids, lengths, labels in data_loader:
        token_ids = token_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(token_ids, lengths)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
# 定义验证流程
def evaluate_model(model, data_loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            token_ids = token_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(token_ids, lengths)
            loss = loss_function(logits, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
# 重新初始化模型，先运行一个训练轮次
torch.manual_seed(random_seed)
generator.manual_seed(random_seed)

textcnn_model = TextCNN(
    vocabulary_size=len(vocabulary),
    embedding_dim=128,
    num_filters=64,
    kernel_size=3,
    padding_index=vocabulary["<PAD>"],
).to(device)
optimizer = torch.optim.Adam(textcnn_model.parameters(), lr=0.001)

train_loss, train_accuracy = train_one_epoch(
    textcnn_model, train_loader, optimizer, loss_function, device,
)
validation_loss, validation_accuracy = evaluate_model(
    textcnn_model, validation_loader, loss_function, device,
)

print(f"Train loss: {train_loss:.4f} | Train accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {validation_loss:.4f} | Validation accuracy: {validation_accuracy:.4f}")

## 多轮训练

重新初始化模型，最多训练 20 轮。根据验证集损失保存最佳模型；若验证损失连续 4 轮没有改善，则提前停止。测试集不参与模型选择。

In [ ]:
# 初始化多轮训练的模型和配置
from sentiment_analysis.paths import CHECKPOINTS_DIR
from sentiment_analysis.sequence_data import TOKEN_PATTERN

model_config = {
    "vocabulary_size": len(vocabulary),
    "embedding_dim": 128,
    "num_filters": 64,
    "kernel_size": 3,
    "padding_index": vocabulary["<PAD>"],
    "dropout": 0.5,
}

torch.manual_seed(random_seed)
generator.manual_seed(random_seed)

textcnn_model = TextCNN(**model_config).to(device)
optimizer = torch.optim.Adam(textcnn_model.parameters(), lr=0.001)

max_epochs = 20
patience = 4
best_validation_loss = float("inf")
epochs_without_improvement = 0
training_history = []

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINTS_DIR / "textcnn_extended_best.pt"

In [ ]:
# 训练模型并按验证损失保存最佳结果
for epoch in range(1, max_epochs + 1):
    train_loss, train_accuracy = train_one_epoch(
        textcnn_model, train_loader, optimizer, loss_function, device,
    )
    validation_loss, validation_accuracy = evaluate_model(
        textcnn_model, validation_loader, loss_function, device,
    )

    training_history.append({
        "EPOCH": epoch,
        "TRAIN LOSS": train_loss,
        "TRAIN ACCURACY": train_accuracy,
        "VALIDATION LOSS": validation_loss,
        "VALIDATION ACCURACY": validation_accuracy,
    })

    print(
        f"Epoch {epoch}: "
        f"Train loss {train_loss:.4f}, accuracy {train_accuracy:.4f} | "
        f"Validation loss {validation_loss:.4f}, accuracy {validation_accuracy:.4f}"
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": textcnn_model.state_dict(),
            "model_config": model_config,
            "vocabulary": vocabulary,
            "max_length": max_length,
            "token_pattern": TOKEN_PATTERN.pattern,
            "lowercase": True,
            "random_seed": random_seed,
            "epoch": epoch,
            "validation_loss": validation_loss,
        }, checkpoint_path)
        print(f"Best model saved: {checkpoint_path}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print("Early stopping: validation loss did not improve.")
            break

training_results = pd.DataFrame(training_history)
display(training_results.round(4))

from sentiment_analysis.paths import TABLES_DIR

TABLES_DIR.mkdir(parents=True, exist_ok=True)
training_results.to_csv(TABLES_DIR / "textcnn_training_history.csv", index=False)

In [ ]:
# 绘制训练与验证曲线
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(training_results["EPOCH"], training_results["TRAIN LOSS"], marker="o", label="Train")
axes[0].plot(training_results["EPOCH"], training_results["VALIDATION LOSS"], marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(training_results["EPOCH"], training_results["TRAIN ACCURACY"], marker="o", label="Train")
axes[1].plot(training_results["EPOCH"], training_results["VALIDATION ACCURACY"], marker="o", label="Validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
axes[1].legend()

figure.tight_layout()

from sentiment_analysis.paths import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "textcnn_training_curves.png", dpi=150)
plt.show()